# NutriVision AI — Exploratory Data Analysis
Food-101 dataset inspection, class distribution, and sample visualisation.

In [ ]:
import sys, json
from pathlib import Path
sys.path.insert(0, str(Path('..').resolve()))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from PIL import Image
from collections import Counter

from configs.config import RAW_DATA_DIR, LOGS_DIR
from src.database import FOOD101_CLASSES, FOOD_NUTRITION
from src.data_pipeline import DataCleaner, get_train_transforms, get_val_transforms
from src.data_pipeline import denormalize
import torch

sns.set_theme(style='whitegrid', palette='Blues_d')
plt.rcParams['figure.dpi'] = 120
print(f'Food-101 classes: {len(FOOD101_CLASSES)}')

## 1. Nutritional Database Overview

In [ ]:
records = []
for name, n in FOOD_NUTRITION.items():
    records.append({'food': name.replace('_',' ').title(), **n})

df = pd.DataFrame(records).sort_values('calories', ascending=False)
print(df.describe().round(2))
df.head(10)

In [ ]:
# Top-20 highest calorie foods
top20_cal = df.head(20)
fig, ax = plt.subplots(figsize=(12, 6))
bars = ax.barh(top20_cal['food'], top20_cal['calories'], color='#2C6FB2', alpha=0.85)
ax.set_xlabel('Calories per 100g', fontsize=12)
ax.set_title('Top 20 Highest Calorie Foods in Dataset', fontsize=14, fontweight='bold')
ax.axvline(df['calories'].mean(), color='#ef4444', linestyle='--', label=f'Mean ({df["calories"].mean():.0f} kcal)')
ax.legend()
for bar, val in zip(bars, top20_cal['calories']):
    ax.text(val+2, bar.get_y()+bar.get_height()/2, f'{val}', va='center', fontsize=8)
plt.tight_layout()
plt.savefig(LOGS_DIR / 'eda_top20_calories.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Macro distribution across all foods
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, col, color, label in [
    (axes[0], 'protein',        '#3b82f6', 'Protein (g/100g)'),
    (axes[1], 'carbohydrates',  '#f59e0b', 'Carbohydrates (g/100g)'),
    (axes[2], 'fat',            '#ef4444', 'Fat (g/100g)'),
]:
    ax.hist(df[col], bins=20, color=color, alpha=0.8, edgecolor='white')
    ax.axvline(df[col].mean(), color='black', linestyle='--', linewidth=1.5, label=f'Mean={df[col].mean():.1f}')
    ax.set_xlabel(label, fontsize=11)
    ax.set_ylabel('Count', fontsize=11)
    ax.set_title(f'{col.title()} Distribution', fontsize=12)
    ax.legend(fontsize=9)
plt.suptitle('Macronutrient Distributions Across Food-101 Classes', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(LOGS_DIR / 'eda_macro_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Correlation heatmap
numeric_cols = ['calories','protein','carbohydrates','fat','fiber','sugar','sodium']
corr = df[numeric_cols].corr()
fig, ax = plt.subplots(figsize=(8, 7))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, square=True, linewidths=0.5, ax=ax,
            cbar_kws={'shrink': 0.8})
ax.set_title('Nutrient Correlation Matrix', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(LOGS_DIR / 'eda_nutrient_correlation.png', dpi=150, bbox_inches='tight')
plt.show()

## 2. Dataset Structure Check (if downloaded)

In [ ]:
data_root = RAW_DATA_DIR / 'food-101' / 'images'
if data_root.exists():
    cleaner = DataCleaner(data_root)
    eda_df  = cleaner.generate_eda_report('train')
    print(eda_df.describe())

    fig, ax = plt.subplots(figsize=(14, 5))
    ax.bar(range(len(eda_df)), eda_df['count'], color='#2C6FB2', alpha=0.8)
    ax.set_xlabel('Class Index', fontsize=11)
    ax.set_ylabel('Image Count', fontsize=11)
    ax.set_title('Class Distribution in Training Set', fontsize=13)
    ax.axhline(eda_df['count'].mean(), color='red', linestyle='--',
               label=f'Mean = {eda_df["count"].mean():.0f}')
    ax.legend()
    plt.tight_layout()
    plt.show()
else:
    print(f'Dataset not found at {data_root}. Download Food-101 to run this cell.')

## 3. Augmentation Visualisation

In [ ]:
# Create a sample image and show augmentation effects
import torchvision.transforms as T

sample_img = Image.new('RGB', (224, 224), color=(180, 120, 60))
train_tf   = get_train_transforms()

fig, axes = plt.subplots(2, 5, figsize=(15, 6))
for i, ax in enumerate(axes.flat):
    aug_tensor = train_tf(sample_img)
    img_vis    = denormalize(aug_tensor).permute(1,2,0).numpy()
    ax.imshow(img_vis)
    ax.set_title(f'Aug {i+1}', fontsize=9)
    ax.axis('off')

plt.suptitle('Training Augmentation Samples (same image, 10 random augmentations)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(LOGS_DIR / 'eda_augmentation_samples.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. BMI Distribution Simulation

In [ ]:
from src.bmi_recommender import classify_bmi

np.random.seed(42)
bmi_samples = np.random.normal(loc=25.5, scale=4.5, size=1000)
bmi_samples = np.clip(bmi_samples, 13, 45)

categories = [classify_bmi(b)['category'] for b in bmi_samples]
cat_counts  = Counter(categories)
colors_map  = {
    'Underweight': '#3B82F6', 'Normal Weight': '#22C55E',
    'Overweight': '#F59E0B', 'Obese Class I': '#EF4444',
    'Obese Class II/III': '#7C3AED'
}

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Histogram
ax = axes[0]
ax.hist(bmi_samples, bins=30, color='#2C6FB2', alpha=0.7, edgecolor='white')
for lo, hi, label, color in [(0,18.5,'',''),(18.5,25,'',''),(25,30,'',''),(30,35,'',''),(35,999,'','')]:
    ax.axvspan(lo, min(hi,45), alpha=0.07, color=list(colors_map.values())[list(colors_map.keys()).index(list(colors_map.keys())[0])])
for val, color, label in [(18.5,'#3B82F6','18.5'),(25,'#22C55E','25'),(30,'#F59E0B','30'),(35,'#EF4444','35')]:
    ax.axvline(val, color=color, linestyle='--', linewidth=1.5, label=label)
ax.set_xlabel('BMI', fontsize=12)
ax.set_ylabel('Count', fontsize=12)
ax.set_title('BMI Distribution (Simulated Population)', fontsize=13)
ax.legend(title='WHO Thresholds', fontsize=9)

# Pie
ax2 = axes[1]
labels_ = list(cat_counts.keys())
sizes_  = list(cat_counts.values())
colors_ = [colors_map.get(l,'#888') for l in labels_]
ax2.pie(sizes_, labels=labels_, colors=colors_, autopct='%1.1f%%',
        startangle=140, pctdistance=0.82, textprops={'fontsize': 9})
ax2.set_title('BMI Category Distribution', fontsize=13)

plt.suptitle('BMI Analysis — Simulated Population (n=1,000)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(LOGS_DIR / 'eda_bmi_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print(dict(cat_counts))